In [1]:
import os
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import gc

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Impor

In [2]:
data_dir = Path('/lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/Raw Data')
samples = []
for h5_file in data_dir.glob('*_norm.h5ad'):
    if h5_file.name == 'adata_CS17.h5ad':
        continue
    sample_name = h5_file.stem.rsplit('_norm', 1)[0]
    samples.append((sample_name, h5_file))
print(f"Found {len(samples)} samples: {[s[0] for s in samples]}")

Found 52 samples: ['section_022', 'section_154', 'section_047', 'section_048', 'section_064', 'section_068', 'section_078', 'section_076', 'section_212', 'section_052', 'section_035', 'section_151', 'section_182', 'section_040', 'section_071', 'section_205', 'section_029', 'section_134', 'section_084', 'section_046', 'section_111', 'section_172', 'section_091', 'section_202', 'section_161', 'section_101', 'section_054', 'section_142', 'section_165', 'section_086', 'section_131', 'section_145', 'section_121', 'section_061', 'section_141', 'section_187', 'section_241', 'section_050', 'section_231', 'section_031', 'section_096', 'section_127', 'section_221', 'section_106', 'section_177', 'section_124', 'section_251', 'section_192', 'section_137', 'section_261', 'section_070', 'section_252']


In [ ]:
for sample_name, h5_path in samples:
    print(f"Processing {sample_name}")
    adata = sc.read_h5ad(h5_path)
    
    # Identify mitochondrial genes
    adata.var['mt'] = adata.var_names.str.startswith('MT-')  
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print(f"Mean pct_counts_mt: {adata.obs['pct_counts_mt'].mean():.2f}%")
    
    # Initial cell count
    initial_cells = adata.n_obs
    print(f"Initial #cells: {initial_cells}")
    
    # Filters
    min_count_thresh = adata.obs["total_counts"].quantile(0.01)
    max_count_thresh = adata.obs["total_counts"].quantile(0.99)
    min_genes_thresh = 200
    
    sc.pp.filter_cells(adata, min_counts=min_count_thresh)
    after_min = adata.n_obs
    print(f"Cells filtered out by min_counts ({min_count_thresh:.0f}): {initial_cells - after_min}")
    
    sc.pp.filter_cells(adata, max_counts=max_count_thresh)
    after_max = adata.n_obs
    print(f"Cells filtered out by max_counts ({max_count_thresh:.0f}): {after_min - after_max}")
    
    sc.pp.filter_cells(adata, min_genes=min_genes_thresh)
    after_min_genes = adata.n_obs
    print(f"Cells filtered out by min_genes ({min_genes_thresh}): {after_max - after_min_genes}")
    
    adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
    after_mt = adata.n_obs
    print(f"Cells filtered out by MT filter (<20%): {after_min_genes - after_mt}")
    print(f"#cells after MT filter: {after_mt}")
    
    # Save
    output_dir = Path('/lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/QC_data')
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{sample_name}_QC_norm.h5ad"
    adata.write(output_path)
    print(f"Saved {sample_name} to {output_path}")
    
    # Memory clearance
    del adata
    gc.collect()

Processing section_022


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Mean pct_counts_mt: 2.47%
Initial #cells: 7557


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Cells filtered out by min_counts (303): 76
Cells filtered out by max_counts (86027): 76


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


... (277 outputs left out)

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Cells filtered out by min_genes (200): 0
Cells filtered out by MT filter (<20%): 0
#cells after MT filter: 8624
Saved section_070 to /lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/QC_data/section_070_QC_norm.h5ad
Processing section_252


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Mean pct_counts_mt: 1.84%
Initial #cells: 431
Cells filtered out by min_counts (2604): 5
Cells filtered out by max_counts (69175): 5
Cells filtered out by min_genes (200): 0
Cells filtered out by MT filter (<20%): 0
#cells after MT filter: 421
Saved section_252 to /lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/QC_data/section_252_QC_norm.h5ad
